# 02 · NMR Fundamentals: From Angles to Observables
### *Making protein structure visible with nuclear magnetic resonance*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elkins-lab/diff-biophys/blob/main/examples/interactive_tutorials/02_nmr_fundamentals.ipynb)

---

**Prerequisites:** Notebook 01 (or comfort with gradient descent basics)

**What you will learn:**
1. What NMR measures and *why* it encodes protein structure
2. **Cα chemical shifts** — how backbone torsion angles predict shift values
3. **The Karplus equation** — measuring dihedral angles through J-couplings
4. **Residual Dipolar Couplings (RDCs)** — long-range orientational restraints
5. How gradients connect NMR observables back to structure

**Time:** ~45 minutes


In [ ]:
%pip install -q diff-biophys==0.1.5 matplotlib
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 12,
})
print("Ready. JAX devices:", jax.devices())

---
## 1 · What is NMR?

**Nuclear Magnetic Resonance (NMR)** exploits the fact that certain atomic nuclei
(¹H, ¹³C, ¹⁵N) behave like tiny magnets.  Put a protein in a strong magnetic field,
pulse it with radio waves, and each nucleus rings back at a slightly different frequency.
That frequency difference — called the **chemical shift** (δ, in ppm) — is
exquisitely sensitive to the local chemical environment.

### The key insight

> The chemical shift of a nucleus depends on the **geometry** of the atoms around it.
> Geometry is encoded in **bond angles and dihedral angles**.
> Therefore: NMR shifts → dihedral angles → protein structure.

`diff-biophys` implements this chain **differentiably**, so you can run gradient descent
from measured shifts all the way back to atomic coordinates.

### NMR observables we will cover

| Observable | Symbol | What it encodes |
|---|---|---|
| Cα chemical shift | δ(Cα) | Local secondary structure (helix / sheet / coil) |
| J-coupling constant | ³J(HN,Hα) | Backbone φ dihedral angle (Karplus equation) |
| Residual Dipolar Coupling | D | Global orientation of bond vectors |


---
## 2 · Cα Chemical Shifts and Secondary Structure

The **Cα chemical shift** (ppm) of a residue shifts away from its random-coil value
depending on whether the residue is in a helix, sheet, or loop:

| Secondary structure | Δδ(Cα) relative to random coil |
|---|---|
| α-helix | **+3.1 ppm** (downfield shift) |
| β-sheet | **−1.5 ppm** (upfield shift) |
| Random coil | 0 ppm |

`predict_ca_shifts` implements this using **soft Gaussian detectors** in (φ, ψ) space,
so the prediction is fully differentiable through the torsion angles.


In [ ]:
from diff_biophys.nmr.chemical_shifts import predict_ca_shifts, RANDOM_COIL_CA

# Random-coil shift for alanine (52.5 ppm)
rc_shift = jnp.array([RANDOM_COIL_CA["ALA"]], dtype=jnp.float32)
print(f"Alanine random-coil Cα shift: {float(rc_shift[0]):.1f} ppm")

# --- Predict shifts across the Ramachandran map ---
phi_grid = jnp.linspace(-jnp.pi, jnp.pi, 60)
psi_grid = jnp.linspace(-jnp.pi, jnp.pi, 60)
phi_mesh, psi_mesh = jnp.meshgrid(phi_grid, psi_grid)

# Vectorise: predict one residue at a time over the grid
phi_flat = phi_mesh.ravel()
psi_flat = psi_mesh.ravel()
rc_flat  = jnp.full_like(phi_flat, RANDOM_COIL_CA["ALA"])

shifts_flat = predict_ca_shifts(phi_flat, psi_flat, rc_flat)
shifts_grid = shifts_flat.reshape(phi_mesh.shape)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.contourf(np.degrees(phi_grid), np.degrees(psi_grid), shifts_grid,
                 levels=30, cmap="RdBu_r")
plt.colorbar(im, ax=ax, label="Predicted δ(Cα)  [ppm]")

# Mark canonical secondary structure regions
ax.scatter([-57], [-47], s=200, marker="*", color="gold", zorder=5,
           label="α-helix (φ=−57°, ψ=−47°)")
ax.scatter([-120], [120], s=200, marker="D", color="lime", zorder=5,
           label="β-strand (φ=−120°, ψ=+120°)")
ax.set_xlabel("φ (degrees)"); ax.set_ylabel("ψ (degrees)")
ax.set_title("Ramachandran map coloured by predicted Cα shift")
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

In [ ]:
# --- Side-by-side: helix vs. sheet vs. coil ---
phi_h = jnp.full((10,), jnp.deg2rad(-57.0),  dtype=jnp.float32)   # helix
psi_h = jnp.full((10,), jnp.deg2rad(-47.0),  dtype=jnp.float32)
phi_s = jnp.full((10,), jnp.deg2rad(-120.0), dtype=jnp.float32)   # sheet
psi_s = jnp.full((10,), jnp.deg2rad(120.0),  dtype=jnp.float32)
phi_c = jnp.full((10,), jnp.deg2rad(-70.0),  dtype=jnp.float32)   # coil
psi_c = jnp.full((10,), jnp.deg2rad(150.0),  dtype=jnp.float32)
rc    = jnp.full((10,), RANDOM_COIL_CA["ALA"], dtype=jnp.float32)

shifts_h = predict_ca_shifts(phi_h, psi_h, rc)
shifts_s = predict_ca_shifts(phi_s, psi_s, rc)
shifts_c = predict_ca_shifts(phi_c, psi_c, rc)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(10) + 1
ax.bar(x - 0.25, shifts_h, 0.25, label="α-helix",   color="steelblue")
ax.bar(x,        shifts_s, 0.25, label="β-strand",  color="crimson")
ax.bar(x + 0.25, shifts_c, 0.25, label="coil",      color="grey", alpha=0.7)
ax.axhline(RANDOM_COIL_CA["ALA"], ls="--", color="k", alpha=0.4, label="random coil")
ax.set_xlabel("Residue"); ax.set_ylabel("δ(Cα)  [ppm]")
ax.set_title("Predicted Cα shifts: helix > coil > sheet")
ax.legend(); plt.tight_layout(); plt.show()

print(f"  Helix mean:  {float(jnp.mean(shifts_h)):.2f} ppm")
print(f"  Sheet mean:  {float(jnp.mean(shifts_s)):.2f} ppm")
print(f"  Coil  mean:  {float(jnp.mean(shifts_c)):.2f} ppm")

---
## 3 · The Karplus Equation

**³J coupling constants** measure how strongly two nuclei interact through three chemical bonds.
For the backbone H–N–Cα–H (HN–Hα) coupling, the Karplus equation gives:

$$J(\phi) = A \cos^2(\phi - 60°) + B \cos(\phi - 60°) + C$$

where $\phi$ is the backbone **phi** torsion angle.

This means: **measure J → infer φ → constrain the backbone geometry.**

The Vuister & Bax (1993) coefficients are: $A = 6.98$, $B = -1.38$, $C = 1.72$ Hz.


In [ ]:
from diff_biophys.nmr.karplus import calculate_karplus_j

A, B, C = 6.98, -1.38, 1.72   # Vuister & Bax 1993

# The coupling depends on phi - 60° (offset convention)
phi_vals = jnp.linspace(-jnp.pi, jnp.pi, 360)
theta    = phi_vals - jnp.deg2rad(60.0)   # offset for HN-Ha coupling
J_vals   = calculate_karplus_j(theta, A, B, C)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(np.degrees(phi_vals), J_vals, "steelblue", lw=2.5)
ax.axhline(0, color="k", lw=0.5)

# Annotate canonical phi values
for phi_deg, label, color in [(-57, "α-helix\nφ=−57°", "gold"),
                               (-120, "β-strand\nφ=−120°", "lime")]:
    phi_r  = jnp.deg2rad(phi_deg)
    theta_r = phi_r - jnp.deg2rad(60.0)
    j0 = float(calculate_karplus_j(jnp.array([theta_r]), A, B, C)[0])
    ax.scatter([phi_deg], [j0], s=120, zorder=5, color=color, edgecolors="k")
    ax.annotate(f"{label}\nJ={j0:.1f} Hz", (phi_deg, j0),
                xytext=(phi_deg + 20, j0 + 0.8), fontsize=9,
                arrowprops=dict(arrowstyle="->", lw=1.2))

ax.set_xlabel("φ (degrees)"); ax.set_ylabel("³J(HN,Hα)  [Hz]")
ax.set_title("Karplus curve: J-coupling as a function of backbone φ")
ax.set_xlim(-180, 180); plt.tight_layout(); plt.show()

print("J-coupling for α-helix  (φ=−57°):", round(float(
    calculate_karplus_j(jnp.array([jnp.deg2rad(-57-60)]), A, B, C)[0]), 2), "Hz")
print("J-coupling for β-strand (φ=−120°):", round(float(
    calculate_karplus_j(jnp.array([jnp.deg2rad(-120-60)]), A, B, C)[0]), 2), "Hz")

### Gradient through Karplus

The gradient $\frac{dJ}{d\phi}$ tells you: *"if I change the backbone angle slightly,
how does the measured J-coupling change?"*

This is the key to NMR-driven structure refinement: if the predicted J doesn't match
the experimental value, the gradient tells you *which direction* to move $\phi$.


In [ ]:
# Compute and visualise dJ/dθ across the full Karplus curve
grad_J = jax.grad(lambda t: jnp.sum(calculate_karplus_j(t, A, B, C)))
theta_vals = phi_vals - jnp.deg2rad(60.0)
dJ_dtheta = grad_J(theta_vals)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax1.plot(np.degrees(phi_vals), J_vals, "steelblue", lw=2)
ax1.set_ylabel("J  [Hz]"); ax1.set_title("Karplus curve and its gradient")

ax2.plot(np.degrees(phi_vals), dJ_dtheta, "crimson", lw=2)
ax2.axhline(0, color="k", lw=0.5)
ax2.set_ylabel("dJ/dθ  [Hz/rad]"); ax2.set_xlabel("φ (degrees)")
ax2.set_xlim(-180, 180)
plt.tight_layout(); plt.show()

---
## 4 · Residual Dipolar Couplings (RDCs)

When a protein is slightly aligned in solution (e.g., in a liquid crystal medium),
dipolar couplings between nuclei are no longer averaged to zero.
The remaining **residual dipolar coupling** (RDC) $D$ depends on the
*orientation* of the bond vector relative to the magnetic field:

$$D_{NH} = D_{\max} \sum_{i,j} v_i S_{ij} v_j$$

where:
- $\mathbf{v}$ = unit vector along the N–H bond
- $\mathbf{S}$ = **Saupe alignment tensor** (a 3×3 symmetric traceless matrix)
- $D_{\max}$ = maximum dipolar coupling (a known constant)

### The magic angle

A bond at $\theta = \arccos(1/\sqrt{3}) \approx 54.74°$ from the alignment axis has
$D = 0$, because $(3\cos^2\theta - 1) = 0$.  This is called the **magic angle**.


In [ ]:
from diff_biophys.nmr.rdc import calculate_rdc_from_tensor

# A simple axially-symmetric Saupe tensor aligned along z
# Szz = 0.1 (weak alignment), Sxx = Syy = -0.05
S = jnp.array([[-0.05, 0.0, 0.0],
               [ 0.0, -0.05, 0.0],
               [ 0.0,  0.0,  0.1]], dtype=jnp.float32)

# Sweep bond orientation from z-axis (θ = 0°) to x-axis (θ = 90°)
theta_vals = jnp.linspace(0, jnp.pi / 2, 180)
# Bond vectors in the xz-plane: v = (sin θ, 0, cos θ)
vecs = jnp.stack([jnp.sin(theta_vals), jnp.zeros_like(theta_vals),
                  jnp.cos(theta_vals)], axis=-1)
rdcs = calculate_rdc_from_tensor(vecs, S, d_max=1.0)

magic = float(jnp.rad2deg(jnp.arccos(1 / jnp.sqrt(3))))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.degrees(theta_vals), rdcs, "steelblue", lw=2.5)
ax.axhline(0, color="k", lw=0.5)
ax.axvline(magic, color="crimson", ls="--", lw=1.5,
           label=f"Magic angle = {magic:.1f}°  (D = 0)")
ax.set_xlabel("θ  (degrees from alignment axis)")
ax.set_ylabel("RDC  (normalised units)")
ax.set_title("RDC angular dependence — $(3\cos^2\theta - 1)$ shape")
ax.legend(); plt.tight_layout(); plt.show()

# Verify the magic angle gives D ≈ 0
magic_vec = jnp.array([[jnp.sin(jnp.deg2rad(magic)), 0.0,
                         jnp.cos(jnp.deg2rad(magic))]], dtype=jnp.float32)
print(f"RDC at magic angle: {float(calculate_rdc_from_tensor(magic_vec, S)[0]):.2e}  (should be ≈ 0)")

---
## 5 · How gradients connect NMR to structure

Now let's see the full pipeline in action.

**Scenario:** We have "experimental" Cα chemical shifts for a 6-residue peptide.
The shifts suggest it's helical ($\delta \approx 55.6$ ppm), but our starting model
has β-strand torsion angles ($\phi = -120°, \psi = +120°$).

**Goal:** Use gradient descent on the shift MSE to nudge the torsion angles toward helix.


In [ ]:
import optax

# --- "Experimental" target: helix-like shifts ---
phi_helix = jnp.full((6,), jnp.deg2rad(-57.0),  dtype=jnp.float32)
psi_helix = jnp.full((6,), jnp.deg2rad(-47.0),  dtype=jnp.float32)
rc        = jnp.full((6,), RANDOM_COIL_CA["ALA"], dtype=jnp.float32)
target_shifts = predict_ca_shifts(phi_helix, psi_helix, rc)

# --- Starting model: beta-strand ---
phi_start = jnp.full((6,), jnp.deg2rad(-120.0), dtype=jnp.float32)
psi_start = jnp.full((6,), jnp.deg2rad( 120.0), dtype=jnp.float32)

# Stack phi and psi as a single (2, 6) parameter array
params = jnp.stack([phi_start, psi_start])

def shift_mse(params):
    phi, psi = params[0], params[1]
    predicted = predict_ca_shifts(phi, psi, rc)
    return jnp.mean((predicted - target_shifts) ** 2)

print(f"Initial MSE:  {shift_mse(params):.4f} ppm²")

# --- Optimise ---
optimizer = optax.adam(0.02)
opt_state = optimizer.init(params)
grad_fn   = jax.jit(jax.value_and_grad(shift_mse))

losses_nmr, phi_trajectory = [], []
for step in range(200):
    loss_val, grads = grad_fn(params)
    losses_nmr.append(float(loss_val))
    phi_trajectory.append(float(jnp.degrees(params[0, 0])))
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

print(f"Final   MSE:  {losses_nmr[-1]:.6f} ppm²")
print(f"phi: {phi_trajectory[0]:.1f}° → {phi_trajectory[-1]:.1f}°  (target: −57°)")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.semilogy(losses_nmr, "steelblue", lw=2)
ax1.set_xlabel("Optimisation step"); ax1.set_ylabel("Shift MSE  [ppm²]")
ax1.set_title("Chemical-shift loss during torsion refinement")

ax2.plot(phi_trajectory, "crimson", lw=2)
ax2.axhline(-57, color="gold", lw=2, ls="--", label="Helix target (φ=−57°)")
ax2.axhline(-120, color="grey", lw=1.5, ls=":", label="Starting point (φ=−120°)")
ax2.set_xlabel("Optimisation step"); ax2.set_ylabel("φ  [degrees]")
ax2.set_title("Backbone φ converging toward helix")
ax2.legend(); plt.tight_layout(); plt.show()

print("\n✅ NMR-driven gradient descent nudged torsions from strand to helix geometry!")

---
## 6 · Summary

| Observable | Function | What it tells you |
|---|---|---|
| Cα chemical shift | `predict_ca_shifts(phi, psi, rc)` | Secondary structure content |
| ³J coupling | `calculate_karplus_j(theta, A, B, C)` | Backbone φ angle |
| RDC | `calculate_rdc_from_tensor(vecs, S)` | Global bond orientations |
| Gradient | `jax.grad(loss)(params)` | Which direction to move torsions |

### What's next?
- **Notebook 03 — CD Spectroscopy**: a completely different physical observable — polarised light — differentiably simulated from atomic positions.

> 💡 Every observable in `diff-biophys` follows the same pattern:
> `forward_model(coords) → observable → loss vs. experiment → gradient → update coords`.
